# 💳 Taller de Clustering — Parte 2: Tarjetas de crédito

Notebook de trabajo: solo contiene las fases ya vistas (ver [`learn_roadmap.md`](learn_roadmap.md)).

## Fase 07 · Comprensión del problema y de los datos

**Objetivo:** conocer los datos de **8.950 clientes** de tarjeta de crédito para, más adelante, **segmentarlos** y diseñar una campaña de marketing para cada grupo.

| | 🍄 Setas | 💳 Tarjetas |
|---|---|---|
| Datos | categóricos (letras) | **numéricos** |
| Etiqueta | sí, solo para validar | **no hay** |
| Cómo se valida | ARI / NMI | **métricas internas + interpretación** |
| Pasos nuevos | One-Hot | **imputar nulos y escalar** |

![El taller: setas y tarjetas](assets/taller/f07-taller.png)

*Fuente: diapositivas del bootcamp «Aprendizaje No Supervisado» (FactoriaF5).*

> 📖 **En los libros**
>
> - **Primer libro:** la **segmentación de clientes** es un uso típico del clustering. Ejemplo: en un blog, sin etiquetas, descubre que el **40 %** de visitantes son adolescentes que leen cómics al salir de clase y el **20 %** adultos que leen ciencia ficción los fines de semana. Con cada grupo se diseña una **estrategia distinta**.
> - **Segundo libro:** la misma idea con votantes: grupos informales como *«madres futboleras»* o *«jubilados aburridos»*. **El nombre del grupo lo pone una persona.**

In [1]:
%matplotlib inline
# Hace que los gráficos aparezcan dentro del notebook
import warnings
warnings.filterwarnings("ignore")  # oculta avisos para que la salida quede limpia

import pandas as pd              # DataFrame: la "hoja de cálculo" con la que trabajamos
import numpy as np               # cálculo numérico
import seaborn as sns            # gráficos estadísticos con poco código
import matplotlib.pyplot as plt  # base de todos los gráficos

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # semilla fija: así los resultados se pueden repetir

df = pd.read_csv("docs/workshop/data/credit_card.csv")
df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


### ¿Qué significa cada columna?

Comportamiento de cada cliente en los **últimos meses** (`TENURE` = meses como cliente):

| Columna | Qué es |
|---|---|
| `CUST_ID` | identificador del cliente |
| `BALANCE` | saldo pendiente en la tarjeta |
| `BALANCE_FREQUENCY` | cada cuánto se actualiza el saldo (0 = nunca, 1 = siempre) |
| `PURCHASES` | total gastado en compras |
| `ONEOFF_PURCHASES` | compras **de una vez** (pago único) |
| `INSTALLMENTS_PURCHASES` | compras **a plazos** |
| `CASH_ADVANCE` | dinero en efectivo sacado con la tarjeta (**adelantos**) |
| `PURCHASES_FREQUENCY` | cada cuánto compra (0 a 1) |
| `ONEOFF_PURCHASES_FREQUENCY` | cada cuánto compra de una vez (0 a 1) |
| `PURCHASES_INSTALLMENTS_FREQUENCY` | cada cuánto compra a plazos (0 a 1) |
| `CASH_ADVANCE_FREQUENCY` | cada cuánto saca adelantos (0 a 1) |
| `CASH_ADVANCE_TRX` | nº de adelantos |
| `PURCHASES_TRX` | nº de compras |
| `CREDIT_LIMIT` | límite de crédito |
| `PAYMENTS` | total pagado |
| `MINIMUM_PAYMENTS` | pagos mínimos realizados |
| `PRC_FULL_PAYMENT` | % de veces que paga **la deuda entera** |
| `TENURE` | meses como cliente |

In [2]:
# ¿Cuántos clientes (filas) y cuántas variables (columnas)?
df.shape

(8950, 18)

In [3]:
# Tipos de datos: ¿hay texto o todo es número?
print(df.dtypes.value_counts())
df.dtypes

float64    14
int64       3
str         1
Name: count, dtype: int64


CUST_ID                                 str
BALANCE                             float64
BALANCE_FREQUENCY                   float64
PURCHASES                           float64
ONEOFF_PURCHASES                    float64
INSTALLMENTS_PURCHASES              float64
CASH_ADVANCE                        float64
PURCHASES_FREQUENCY                 float64
ONEOFF_PURCHASES_FREQUENCY          float64
PURCHASES_INSTALLMENTS_FREQUENCY    float64
CASH_ADVANCE_FREQUENCY              float64
CASH_ADVANCE_TRX                      int64
PURCHASES_TRX                         int64
CREDIT_LIMIT                        float64
PAYMENTS                            float64
MINIMUM_PAYMENTS                    float64
PRC_FULL_PAYMENT                    float64
TENURE                                int64
dtype: object

Todo es **número** salvo `CUST_ID`. ¿Sirve `CUST_ID` para agrupar? Comprobamos cuántos valores distintos tiene:

In [4]:
print("Valores distintos en CUST_ID:", df["CUST_ID"].nunique(), "de", len(df), "filas")
df["CUST_ID"].head()

Valores distintos en CUST_ID: 8950 de 8950 filas


0    C10001
1    C10002
2    C10003
3    C10004
4    C10005
Name: CUST_ID, dtype: str

Un valor **distinto por cliente**: es solo una **matrícula**, no describe cómo usa la tarjeta. Lo quitaremos en la fase 08.

### Estadística descriptiva: escalas y sesgo

In [5]:
# .T gira la tabla: una fila por variable, más fácil de leer
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.47,2081.53,0.00,128.28,873.39,2054.14,19043.14
BALANCE_FREQUENCY,8950.0,0.88,0.24,0.00,0.89,1.00,1.00,1.00
PURCHASES,8950.0,1003.20,2136.63,0.00,39.64,361.28,1110.13,49039.57
ONEOFF_PURCHASES,8950.0,592.44,1659.89,0.00,0.00,38.00,577.40,40761.25
INSTALLMENTS_PURCHASES,8950.0,411.07,904.34,0.00,0.00,89.00,468.64,22500.00
CASH_ADVANCE,8950.0,978.87,2097.16,0.00,0.00,0.00,1113.82,47137.21
PURCHASES_FREQUENCY,8950.0,0.49,0.40,0.00,0.08,0.50,0.92,1.00
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.20,0.30,0.00,0.00,0.08,0.30,1.00
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.36,0.40,0.00,0.00,0.17,0.75,1.00
CASH_ADVANCE_FREQUENCY,8950.0,0.14,0.20,0.00,0.00,0.00,0.22,1.50


**Dos cosas que saltan a la vista:**

1. **Escalas muy distintas:** `CREDIT_LIMIT` llega a **30.000** y las `*_FREQUENCY` van de **0 a 1**. En una distancia, las grandes **aplastarían** a las pequeñas → habrá que **escalar** (fase 08).
2. **Media ≫ mediana (50 %):** en `CASH_ADVANCE` la media es **979** pero la mediana es **0**: la mayoría no saca efectivo y unos pocos sacan muchísimo. Son **colas largas**.

> 📖 **En los libros**
>
> - **Primer libro:** primer vistazo a las **viviendas de California** (20.640 distritos): `info()` muestra que `total_bedrooms` tiene solo 20.433 valores → **207 nulos**, que rellena con la **mediana** (434). Y avisa de las escalas: `total_rooms` va de **6 a 39.320** y `median_income` de **0,5 a 15** → hay que escalar.
> - **Segundo libro:** si mides la altura en **pulgadas o en centímetros**, las distancias entre personas **cambian**: por eso estandariza cada columna a media 0 y desviación 1.

### ¿Hay valores imposibles?

Como con los `'?'` de las setas, **no nos fiamos**: buscamos de forma sistemática. Con números, «imposible» es un valor **negativo** (no hay compras negativas) o una **frecuencia mayor que 1**.

In [6]:
numericas = df.drop(columns="CUST_ID")
print("Valores negativos en todo el dataset:", (numericas < 0).sum().sum())

frecuencias = [c for c in df.columns if c.endswith("_FREQUENCY")]
print("\nFrecuencias mayores que 1:")
print((df[frecuencias] > 1).sum())

Valores negativos en todo el dataset: 0

Frecuencias mayores que 1:
BALANCE_FREQUENCY                   0
PURCHASES_FREQUENCY                 0
ONEOFF_PURCHASES_FREQUENCY          0
PURCHASES_INSTALLMENTS_FREQUENCY    0
CASH_ADVANCE_FREQUENCY              8
dtype: int64


In [7]:
# ¿Quiénes son esos clientes con frecuencia > 1?
df.loc[df["CASH_ADVANCE_FREQUENCY"] > 1,
       ["CUST_ID", "CASH_ADVANCE_FREQUENCY", "CASH_ADVANCE_TRX", "TENURE"]]

,CUST_ID,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,TENURE
681,C10708,1.250000,12,8
1626,C11680,1.166667,24,6
2555,C12629,1.125000,12,8
2608,C12684,1.100000,20,10
3038,C13127,1.500000,18,6
3253,C13347,1.166667,10,6
8055,C18273,1.090909,28,11
8365,C18588,1.142857,26,7


**8 clientes** tienen `CASH_ADVANCE_FREQUENCY` > 1, algo imposible para una frecuencia. **Todos** llevan menos de 12 meses (`TENURE` 6–11), cuando el **85 %** de los clientes lleva 12: probablemente la frecuencia se calculó mal para clientes nuevos. Son muy pocos (0,09 %): lo anotamos y decidiremos en la fase 08.

### Nulos

In [8]:
# Nulos por columna (solo las que tengan alguno)
nulos = df.isnull().sum()
nulos[nulos > 0]

CREDIT_LIMIT          1
MINIMUM_PAYMENTS    313
dtype: int64

> **Resumen fase 07:** **8.950 clientes × 18 columnas**, todas **numéricas** salvo `CUST_ID` (una **matrícula**: un valor distinto por cliente, no sirve para agrupar). **No hay etiqueta**. Escalas **muy distintas** (0–1 frente a miles) y **colas largas** (media ≫ mediana). Nulos: `CREDIT_LIMIT` (**1**) y `MINIMUM_PAYMENTS` (**313**). Búsqueda sistemática: 0 negativos y **8 frecuencias > 1** (imposibles).